In [53]:
import json

from kafka import KafkaProducer

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=lambda message: json.dumps(message).encode('utf-8')
)

producer.bootstrap_connected()

True

In [54]:

import pandas as pd

url = "https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_2019-10.csv.gz"
columns = ['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'total_amount']
df = pd.read_csv(
    url,
    usecols=columns,
    parse_dates=["lpep_pickup_datetime", "lpep_dropoff_datetime"],
)
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,total_amount
0,2019-10-01 00:26:02,2019-10-01 00:39:58,112,196,1.0,5.88,19.30
1,2019-10-01 00:18:11,2019-10-01 00:22:38,43,263,1.0,0.80,9.05
2,2019-10-01 00:09:31,2019-10-01 00:24:47,255,228,2.0,7.50,22.80
3,2019-10-01 00:37:40,2019-10-01 00:41:49,181,181,1.0,0.90,6.80
4,2019-10-01 00:08:13,2019-10-01 00:17:56,97,188,1.0,2.52,13.56


In [55]:
import importlib
import models

models = importlib.reload(models)
ride = models.ride_from_row(df.iloc[1])
ride

Ride(PULocationID=43, DOLocationID=263, passenger_count=1, trip_distance=0.8, total_amount=9.05, lpep_pickup_datetime=1569889091000, lpep_dropoff_datetime=1569889358000)

In [56]:
from time import perf_counter
import pandas as pd

def to_int(value, default=0):
    return default if pd.isna(value) else int(value)

def to_float(value, default=0.0):
    return default if pd.isna(value) else float(value)

def to_epoch_ms(value, default=0):
    return default if pd.isna(value) else int(value.timestamp() * 1000)

topic_name = 'green-trips'

t0 = perf_counter()

for _, row in df.iterrows():
    message = {
        'lpep_pickup_datetime': to_epoch_ms(row['lpep_pickup_datetime']),
        'lpep_dropoff_datetime': to_epoch_ms(row['lpep_dropoff_datetime']),
        'PULocationID': to_int(row['PULocationID']),
        'DOLocationID': to_int(row['DOLocationID']),
        'passenger_count': to_int(row['passenger_count']),
        'trip_distance': to_float(row['trip_distance']),
        'total_amount': to_float(row['total_amount']),
    }
    producer.send(topic_name, value=message)

producer.flush()

t1 = perf_counter()
took = t1 - t0

In [57]:
print(f'Took {took:.3f} seconds to send {len(df)} messages and flush')

Took 49.863 seconds to send 476386 messages and flush
